In [1]:
import json
import torch
import random
from datasets import load_dataset, concatenate_datasets
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

# Force garbage collection
import gc
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 1. CONFIGURATION
# ==========================================
MODEL_NAME = "google/flan-t5-base"
OUTPUT_DIR = "./final_student_model"
MAX_LENGTH = 512
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)

print(f"Loading tokenizer and model: {MODEL_NAME}...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# ==========================================
# 2. DATA FORMATTING
# ==========================================
def format_nli(example):
    label_map = {'e': 'entailment', 'n': 'neutral', 'c': 'contradiction'}
    target_text = label_map.get(example['label'], 'neutral')
    input_text = f"nli premise: {example['context']} hypothesis: {example['hypothesis']}"
    return {"input_text": input_text, "target_text": target_text}

def format_ie(example):
    sentence = example['sentence']
    ner_list = [f"({span}, {etype})" for span, etype in example['ner']]
    ner_str = ", ".join(ner_list) if ner_list else "None"
    rel_list = [f"({s1}, {r}, {s2})" for s1, r, s2 in example['rel']]
    rel_str = ", ".join(rel_list) if rel_list else "None"
    target_text = f"Entities: {ner_str} | Relations: {rel_str}"
    return {"input_text": f"extract information: {sentence}", "target_text": target_text}

def format_summ(example):
    return {"input_text": f"summarize: {example['dialogue']}", "target_text": example['summary']}

def format_sentiment(example):
    label_map = {0: 'anger', 1: 'joy', 2: 'optimism', 3: 'sadness'}
    return {"input_text": f"classify emotion: {example['text']}", "target_text": label_map[example['label']]}

def format_csqa(example):
    choices = example['choices']
    choices_str = " ".join([f"({l}) {t}" for l, t in zip(choices['label'], choices['text'])])
    try:
        answer_idx = choices['label'].index(example['answerKey'])
        correct_text = choices['text'][answer_idx]
    except ValueError:
        correct_text = ""
    return {"input_text": f"commonsense question: {example['question']} choices: {choices_str}", "target_text": correct_text}

# ==========================================
# 3. LOAD DATASETS (SCALED DOWN)
# ==========================================
print("Loading datasets...")

# --- UPDATE: Limit Local Files to 1000 rows each ---
# We use split='train[:1000]' to slice the local files immediately
data_nli = load_dataset("json", data_files="train.jsonl", split="train[:1000]").map(format_nli, remove_columns=["context", "hypothesis", "label", "uid", "reason", "genre", "emturk", "tag", "model_label"])
data_ie = load_dataset("json", data_files="train_2.jsonl", split="train[:1000]").map(format_ie, remove_columns=["doc_id", "sentence", "ner", "rel", "rel_plus"])

# --- Web Datasets: Reduced to 1000 each to match ---
data_summ = load_dataset("knkarthick/dialogsum", split="train[:1000]").map(format_summ, remove_columns=["id", "dialogue", "summary", "topic"])
data_sent = load_dataset("cardiffnlp/tweet_eval", "emotion", split="train[:1000]").map(format_sentiment, remove_columns=["text", "label"])
data_csqa = load_dataset("tau/commonsense_qa", split="train[:1000]").map(format_csqa, remove_columns=["id", "question", "question_concept", "choices", "answerKey"])

combined_dataset = concatenate_datasets([data_nli, data_ie, data_summ, data_sent, data_csqa])
combined_dataset = combined_dataset.shuffle(seed=SEED)

# --- FILTER BAD ROWS ---
def filter_bad_rows(example):
    return len(example["input_text"]) > 0 and len(example["target_text"]) > 0
combined_dataset = combined_dataset.filter(filter_bad_rows)

print(f"Total Combined Examples: {len(combined_dataset)}")
# Should be ~5000 examples. With batch size 16 (2*8), 1 epoch = ~312 steps.
# 4 epochs ≈ 1248 steps. Perfect for Colab.

# ==========================================
# 4. TOKENIZATION
# ==========================================
print("Splitting & Tokenizing...")
dataset_splits = combined_dataset.train_test_split(test_size=0.1)
train_dataset = dataset_splits["train"]
eval_dataset = dataset_splits["test"]

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")
    labels = tokenizer(examples["target_text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

# ==========================================
# 5. TRAINING SETUP (HARD LIMIT ENABLED)
# ==========================================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # --- HARD LIMIT ---
    max_steps=1200,            # <--- This guarantees training stops at 1200 steps
    # ------------------

    eval_strategy="steps",
    eval_steps=200,            # Check stats every 200 steps
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    learning_rate=1e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    gradient_checkpointing=True,
    fp16=False,

    # Set epochs high; 'max_steps' will override it anyway
    num_train_epochs=10,

    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ==========================================
# 6. RUN
# ==========================================
print("Starting training (capped at 1200 steps)...")
trainer.train()

print(f"Saving to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Loading tokenizer and model: google/flan-t5-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
200,0.055800,0.038065
400,0.034500,0.026853
600,0.028700,0.025327
800,0.029900,0.024046
1000,0.029100,0.023776
1200,0.023700,0.023603


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Saving to ./final_student_model...


('./final_student_model/tokenizer_config.json',
 './final_student_model/special_tokens_map.json',
 './final_student_model/spiece.model',
 './final_student_model/added_tokens.json')

In [5]:
from google.colab import files
files.download("/content/final_student_model")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>